# Verifier-driven MCRDR learning: VAREX trial

This notebook runs one small learning trial end to end:

1. The **worker** (Qwen3.7 Flash) does 10 VAREX practice documents with the MCRDR-selected skill (reduced arm only).
2. Passing attempts become "worked fine" examples (cornerstones) in a copy of the PDF skill tree.
3. The **inspector** (Qwen3.7 Plus) reads each failed attempt, finds the cause, and must prove it by fixing the
   answer and re-scoring it with the official scorer.
4. Proven, skill-related causes become new MCRDR rules. The engine refuses any rule that would change advice for
   the passing examples. Each rule is then re-tested by re-running the failed document and a few passing ones.
5. A results table shows every failure, what was blamed, the rule proposed, whether it was kept, and the cost.

All models run on OpenRouter (provider `alibaba`). The yes/no **checker** for new rule conditions is Qwen3.7 Flash.
Models and limits are in `examples/learning/varex.learner.json`.

**Cost.** Paid cells are marked. Rough guesses, not measurements: collection about $0.20; the inspector up to about
$0.30 per failed document; re-testing about $0.20 per rule. Expect a few dollars at most for this trial.

**How to run it**

1. In VS Code, open this notebook with the **Google Colab** extension: **Select Kernel → Colab**, start a server
   (a CPU runtime is enough).
2. Run the cells in order. Each paid cell can be re-run after a disconnect: finished work is kept and resumed.

Results stay on the runtime under `/content/rdr-skills-mcrdr/experiments/`; the last cell downloads a zip.
Diagnoses contain the correct answers and episodes contain document text, so keep them private.

In [ ]:
# Settings. Change them only deliberately; a new RUN_NAME starts a fresh trial.
BRANCH = "verifier-mcrdr"
WORKDIR = "/content/rdr-skills-mcrdr"
EXPERIMENT = "qwen-dev30"  # examples/agent/varex.mcrdr.qwen-dev30.json (worker model and limits)
DOCUMENTS = 10  # the first N of the 30 dev documents
RUN_NAME = "learn-qwen-dev10"

## 1. Model key

In [ ]:
import getpass
import os

if not os.environ.get("MODEL_API_KEY"):
    key = None
    try:  # Colab Secrets, if this notebook has been granted access to a MODEL_API_KEY secret
        from google.colab import userdata
        key = userdata.get("MODEL_API_KEY")
    except Exception:
        pass
    os.environ["MODEL_API_KEY"] = key or getpass.getpass("OpenRouter key (hidden, not saved): ")
print("MODEL_API_KEY is set" if os.environ.get("MODEL_API_KEY") else "MODEL_API_KEY is missing")

## 2. Setup script

Fetches `scripts/setup_mcrdr_colab.py` from the branch. It needs only the standard library.

In [ ]:
import importlib.util
import sys
import urllib.request
from pathlib import Path

SCRIPT = Path("/content/setup_mcrdr_colab.py")
urllib.request.urlretrieve(
    f"https://raw.githubusercontent.com/dmw2101/RDR-skills-/{BRANCH}/scripts/setup_mcrdr_colab.py", SCRIPT)
spec = importlib.util.spec_from_file_location("setup_mcrdr_colab", SCRIPT)
setup_mcrdr_colab = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = setup_mcrdr_colab
spec.loader.exec_module(setup_mcrdr_colab)
EXPERIMENTS_DIR = Path(WORKDIR) / "experiments"
LEARNER_CONFIG = EXPERIMENTS_DIR / "configs" / f"{RUN_NAME}.learner.json"
LEARNING_DIR = EXPERIMENTS_DIR / "learning" / RUN_NAME
PYTHON = Path(WORKDIR) / ".venv" / "bin" / "python"


def collect(*, execute=True):
    """Set up (idempotently) and, with execute=True, run, inspect and summarize the reduced-arm run."""
    args = ["--experiment", EXPERIMENT, "--limit", str(DOCUMENTS), "--conditions", "reduced",
            "--run-name", RUN_NAME, "--workdir", WORKDIR, "--branch", BRANCH, "--learn"]
    if (EXPERIMENTS_DIR / "results" / RUN_NAME).exists():
        args.append("--resume")  # keep finished episodes; never start a second copy of the run
    if execute:
        args.append("--run")
    if setup_mcrdr_colab.main(args):
        raise RuntimeError(f"{RUN_NAME} stopped; read the message above, fix it, and rerun this cell")
    print(f"\n{RUN_NAME}: {'collected and summarized' if execute else 'set up and checked'}")


def learn(stage):
    """Run one trace_learner stage; each stage saves its output and resumes after an interruption."""
    if not LEARNER_CONFIG.exists():
        raise RuntimeError("run the setup cell first; it writes the learner config")
    setup_mcrdr_colab.Runner().run([PYTHON, "-m", "trace_learner", stage, "--config", LEARNER_CONFIG],
                                   cwd=Path(WORKDIR))

## 3. Set up and check, without model calls

Installs the PDF tools and Python packages, clones the branch, prepares the documents, freezes the PDF skill
snapshot, starts the MCRDR skill service (port 8766), writes the run and learner configs, and runs `doctor`.
Nothing is spent here.

In [ ]:
collect(execute=False)

## 4. Collect: the worker does the documents (paid)

In [ ]:
collect()

## 5. Starting examples (free)

Scores every attempt and copies the PDF skill to `versions/v0`, with the advice each passing attempt saw stored as
cornerstones.

In [ ]:
learn("init")

## 6. Inspector: diagnose each failure (paid)

Prints one line per failed attempt: what the inspector blamed and whether its fix was proven by a passing re-score.

In [ ]:
learn("verify")

## 7. Combine suggestions (paid only when several aim at the same rule)

In [ ]:
learn("consolidate")

## 8. Add rules and re-test them (paid)

Each proposed rule is added to a copy of the tree, checked by the engine, then re-tested by re-running the failed
document and up to 3 passing ones, twice each. Kept rules become `versions/v1`, `v2`, ...

In [ ]:
learn("insert")

## 9. Results

In [ ]:
from IPython.display import Markdown, display

learn("report")
display(Markdown((LEARNING_DIR / "report.md").read_text(encoding="utf-8")))

## 10. Download the results

In [ ]:
import zipfile

ARCHIVE = Path(f"/content/{RUN_NAME}.zip")
with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(EXPERIMENTS_DIR.rglob("*")):
        relative = path.relative_to(EXPERIMENTS_DIR)
        keep = (relative.parts[:2] in (("results", RUN_NAME), ("learning", RUN_NAME))
                or relative.parts[0] in ("configs", "logs") or relative.name == "pdf-snapshot-mcrdr.json")
        if path.is_file() and keep:
            archive.write(path, relative.as_posix())
print(f"Wrote {ARCHIVE} ({ARCHIVE.stat().st_size / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(str(ARCHIVE))
except Exception:
    print("Automatic download is unavailable here; fetch the file from the runtime's file browser.")